In [1]:
import pandas as pd

# 1. Veriyi okuma ve 'Date' sütununu zaman indeksine (datetime) çevirme
df = pd.read_csv("YAP471_Cleaned_Close_Prices.csv")
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# 2. Z-Score Hesaplaması
# 20 günlük kayan (rolling) pencere üzerinden ortalama ve standart sapma
ma_20 = df.rolling(window=20).mean()
std_20 = df.rolling(window=20).std()

# Formülü uygulayarak Z-Score matrisini elde etme
z_scores = (df - ma_20) / std_20

# İlk 25 güne ve son günlere bakalım
print(z_scores.head(25))

                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-03-15       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-16       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-17       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-18       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-19       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-22       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-23       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-24       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-25       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-26       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-29       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-30       NaN       NaN       NaN       NaN       NaN       NaN
2021-0

In [2]:
# Z-skorlarını -1 ile çarparak Beklenen Getiri Proxy'sine çeviriyoruz
expected_returns_proxy = z_scores * -1

# Sorunsuz kullanması için ilk 19 gündeki NaN (boş) verileri atıyoruz
expected_returns_proxy.dropna(inplace=True)

# Çıktıyı yeni ve profesyonel bir isimle kaydediyoruz
expected_returns_proxy.to_csv("YAP471_ZScore_Expected_Returns.csv")

print("Gidecek verinin ilk 5 günü:")
print(expected_returns_proxy.head())

Gidecek verinin ilk 5 günü:
                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-04-12 -1.794911 -1.842845 -2.102636 -1.484519 -1.316402 -2.519826
2021-04-13 -2.136748 -1.832478 -1.932636 -1.411248 -2.763378 -2.460098
2021-04-14 -1.449593 -1.356362 -1.236010 -1.159143 -1.659065 -1.722521
2021-04-15 -1.702196 -1.542771 -1.424865 -1.411819 -1.675836 -2.151362
2021-04-16 -1.455112 -1.484967 -1.416907 -1.249891 -1.497823 -1.694955


Projemizde sinyal üretimi aşamasında, ortalamaya dönüş (mean reversion) stratejisine temel oluşturması amacıyla 20 günlük kayan Z-skoru (rolling Z-score) kullanılmıştır. Sinyal hesaplaması $z_t = (P_t - MA_{20}) / \sigma_{20}$ formülüyle gerçekleştirilmiştir. Modelin doğası gereği, ilk 19 işlem gününde geriye dönük 20 günlük veri seti tam olarak oluşmadığı için hesaplama yapılamamış ve bu günler veri setinde boş (NaN) bırakılarak analiz dışı tutulmuştur. Bu yaklaşım, sinyal inşası sırasında geleceği görme hatasından (look-ahead bias) kesinlikle kaçınılmasını sağlamış ve geriye dönük testin temiz bir kayan örneklem dışı (rolling out-of-sample) tasarımla ilerlemesini güvence altına almıştır.

Stratejimizde Z-skoru hesaplanırken 20 günlük kayan pencere (rolling window) kullanılmıştır. 'Look-ahead bias' (geleceği görme hatası) oluşmaması için ilk 19 günlük veri setten çıkarılmıştır. Markowitz optimizer'ının düşük Z-skorlu (ucuz) hisselere ağırlık verebilmesi için, Z-skoru değerleri -1 ile çarpılarak 'Beklenen Getiri' (Expected Return Proxy) formatına dönüştürülmüştür.